# E-commerce Sales Analytics — Exploratory Data Analysis

**Dataset:** Olist Brazilian E-commerce (public, CC BY-NC-SA 4.0) — 99,441 orders placed Sep 2016 – Oct 2018, stored as 8 relational tables (customers, orders, order items, payments, reviews, products, sellers, category translations).

**Goal:** understand what drives revenue, how delivery performance affects customer satisfaction, and where the business should focus next.

All loading/cleaning lives in `src/ecommerce.py`, so the notebooks, the SQL queries and the Streamlit dashboard analyse the exact same tables.

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import plotly.express as px
from ecommerce import load_all, build_order_facts, build_item_facts

tables = load_all("../data")
facts = build_order_facts("../data")   # one row per order
items = build_item_facts("../data")    # one row per order line item
delivered = facts[facts["order_status"] == "delivered"].copy()
print("order facts:", facts.shape, "| item facts:", items.shape)

## 1. Dataset overview

In [ ]:
for name, df in tables.items():
    print(f"{name:22s} {df.shape[0]:>7,} rows x {df.shape[1]} cols")
print()
print("purchase window:", facts["order_purchase_timestamp"].min(),
      "->", facts["order_purchase_timestamp"].max())
print("\norder status:")
print(facts["order_status"].value_counts())

## 2. Revenue trend

Revenue = `payment_value` summed over **delivered** orders. The trend chart starts at 2017-01: 2016 contains only a sparse test period (266 orders), so including it would distort the picture.

In [ ]:
monthly = (delivered[delivered["order_month"] >= "2017-01"]
           .groupby("order_month")["total_payment"].sum())
print(f"Total revenue (delivered): R$ {delivered['total_payment'].sum():,.0f}")
print(f"Peak month: {monthly.idxmax()} — R$ {monthly.max():,.0f} "
      f"(+{(monthly.max()/monthly.shift(1)[monthly.idxmax()]-1)*100:.1f}% MoM)")
fig = px.line(x=monthly.index, y=monthly.values, markers=True,
              labels={"x": "Month", "y": "Revenue (R$"},
              title="Monthly revenue — delivered orders")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 3. Category performance

Which categories bring the money? (Revenue at item level, delivered orders only.)

In [ ]:
cat = (items[items["order_status"] == "delivered"]
       .groupby("category_en")
       .agg(revenue=("price", "sum"), orders=("order_id", "nunique"))
       .sort_values("revenue", ascending=False))
top10 = cat.head(10)
print(top10.assign(revenue=top10["revenue"].map(lambda v: f"R$ {v:,.0f}")))
print(f"\nTop-3 categories share of revenue: "
      f"{cat['revenue'].head(3).sum()/cat['revenue'].sum():.1%}")
fig = px.bar(top10[::-1], x="revenue", y=top10[::-1].index, orientation="h",
             labels={"revenue": "Revenue (R$)", "category_en": "Category"},
             title="Top 10 categories by revenue")
fig.show()

## 4. Geography — a concentrated business

In [ ]:
st = (delivered.groupby("customer_state")
      .agg(revenue=("total_payment", "sum"), orders=("order_id", "nunique"))
      .sort_values("revenue", ascending=False))
print(st.head(10).assign(revenue=lambda d: d["revenue"].map(lambda v: f"R$ {v:,.0f}")))
print(f"\nSao Paulo share of revenue: {st['revenue'].iloc[0]/st['revenue'].sum():.1%}")
fig = px.bar(st.head(12)[::-1], x="revenue", y=st.head(12)[::-1].index,
             orientation="h", title="Revenue by customer state (top 12)",
             labels={"revenue": "Revenue (R$)", "customer_state": "State"})
fig.show()

## 5. Delivery performance

`delay_days` = actual delivery date − estimated delivery date (whole days; > 0 = late).

In [ ]:
dd = delivered.dropna(subset=["delay_days"])
print(f"Late deliveries: {dd['is_late'].mean():.1%} "
      f"({dd['is_late'].sum():,} of {len(dd):,} delivered orders)")
print(f"Avg days late (when late): {dd.loc[dd['is_late'], 'delay_days'].mean():.1f}")
print(f"Avg days early (when on time): {dd.loc[~dd['is_late'], 'delay_days'].mean():.1f}")
fig = px.histogram(dd, x="delay_days", nbins=60, range_x=(-40, 40),
                   title="Delivery delay distribution (days; > 0 = late)",
                   labels={"delay_days": "Delay (days)"})
fig.show()

## 6. Late deliveries destroy review scores

In [ ]:
rev = delivered.dropna(subset=["review_score", "delay_days"]).copy()
rev["bucket"] = pd.cut(rev["delay_days"], [-10_000, 0, 7, 10_000],
                        labels=["on time / early", "1-7 days late", "8+ days late"])
g = rev.groupby("bucket", observed=True)["review_score"].agg(
    ["mean", "count", lambda s: (s == 5).mean(), lambda s: (s == 1).mean()])
g.columns = ["avg_review", "orders", "share_5_star", "share_1_star"]
print(g.round(2))
fig = px.bar(g.reset_index(), x="bucket", y="avg_review",
             title="Average review score by delivery punctuality",
             labels={"bucket": "", "avg_review": "Avg review score"})
fig.show()

## 7. How customers pay

In [ ]:
pm = delivered.groupby("payment_type").agg(orders=("order_id", "nunique"),
                                             avg_inst=("max_installments", "mean"))
pm["share"] = pm["orders"] / pm["orders"].sum()
print(pm.round(2))
fig = px.pie(pm.reset_index(), names="payment_type", values="orders",
             title="Delivered orders by payment type")
fig.show()

## 8. Key takeaways

Every number below was computed from the data in the cells above:

- **R$ 15.4M revenue** from 96,478 delivered orders (97% of 99,441 placed); **AOV R$ 159.85**. Nov 2017 was the peak month (R$ 1.15M, +53.6% MoM — Black Friday effect).
- **Top categories:** health & beauty (R$ 1.23M), watches & gifts (R$ 1.17M), bed/bath/table (R$ 1.02M); the top 3 are only 26% of revenue — the catalogue is diversified.
- **Geography is concentrated:** São Paulo alone is 37% of revenue; the top 3 states (SP/RJ/MG) are 63%.
- **Only 3.0% of customers ever buy again** — this is an acquisition-driven marketplace, not a retention business. Repeat buyers are worth ~1.9x (R$ 309 vs R$ 161 lifetime value).
- **Delivery is the #1 satisfaction lever:** 6.8% of orders arrive late (avg 10.6 days late); late orders average **2.27★** vs **4.29★** on time, and 54% of late orders get 1 star.
- **Payments:** 77% credit card (avg 3.5 instalments — instalment culture matters in Brazil), 20% boleto (cash-based, 1 instalment).
- **Data-quality notes:** `review_id` is not unique in the raw file (789 duplicated ids); ~1,400 order items have no product category mapped.